# LensRank experiment notebook

The LensRank notebook configures training and validation. The cells below run training only when explicitly executed.


## Environment

Use a Python environment with requirements.txt installed. Start this notebook from LensRank or its parent folder.


In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd().resolve()
if not (PROJECT / 'resumable_experiments.py').exists():
    PROJECT = (PROJECT / 'LensRank').resolve()
if not (PROJECT / 'resumable_experiments.py').exists():
    raise FileNotFoundError('Run this notebook from the LensRank folder or its parent.')

DATA_DIR = PROJECT / 'data'
RUN_DIR = PROJECT / 'artifacts' / 'fresh_runs'
sys.path.insert(0, str(PROJECT))

import pandas as pd
import matplotlib.pyplot as plt
import torch
from resumable_experiments import SuiteConfig, RetrievalSuite, EXPERIMENTS

print('Project:', PROJECT)
print('Data:', DATA_DIR)
print('Artifacts:', RUN_DIR)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 2. Configuration

All inputs and outputs stay inside this project folder: `data/` for dataset files and `artifacts/fresh_runs/` for checkpoints, histories, plots, and comparison CSVs. Set `fast_dev_run=True` only for a smoke test and use a separate project-local `run_dir`. Keep it `False` for the final benchmark.


In [ ]:
cfg = SuiteConfig(
    data_dir=str(DATA_DIR),
    run_dir=str(RUN_DIR),
    epochs=10,
    batch_size=128,
    eval_batch_size=256,
    checkpoint_every_batches=100,
    seed=42,
    fast_dev_run=False,
)
display(pd.DataFrame(EXPERIMENTS).T)


## 3. Validate data

Suite creation fails early when required tables or images are missing.

In [ ]:
suite=RetrievalSuite(cfg)
print(f'Train pairs: {len(suite.train):,}')
print(f'Validation pairs: {len(suite.val):,}')
print('Run directory:',suite.runs)

## 4. Train or resume every experiment

This is the long-running cell. After an interruption, rerun it. The active experiment resumes from its batch checkpoint and completed experiments print `[skip]`.

In [ ]:
comparison=suite.run_all()
display(comparison)

## 5. Fresh comparison

This table is built exclusively from newly completed experiment records. Mean bidirectional Recall is the primary metric.

In [ ]:
results_path = RUN_DIR / 'fresh_comparison.csv'
fresh = pd.read_csv(results_path).sort_values('mean_score', ascending=False)
cols = ['experiment','strategy','best_epoch','i2t_R@1','i2t_R@5','i2t_R@10','t2i_R@1','t2i_R@5','t2i_R@10','mean_score','trainable_parameters']
display(fresh[cols])


In [ ]:
ax = fresh.sort_values('mean_score').plot.barh(x='experiment', y=['i2t_score','t2i_score'], figsize=(10,6), color=['#4C78A8','#F58518'])
ax.set(title='Fresh validation comparison', xlabel='Mean Recall@1/5/10', ylabel='')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
plt.savefig(RUN_DIR / 'fresh_comparison.png', dpi=160, bbox_inches='tight')
plt.show()


## 6. Automatic conclusion

The narrative is computed from the fresh result table so it remains correct if a different strategy wins.

In [ ]:
winner=fresh.iloc[0]
print(f'Best fresh model: {winner.experiment}')
print(f'Mean score: {winner.mean_score:.4f}')
print(f'I2T: {winner.i2t_score:.4f} | T2I: {winner.t2i_score:.4f}')
print(f'Best epoch: {int(winner.best_epoch)} | Trainable parameters: {int(winner.trainable_parameters):,}')

## Recovery notes

- Kernel/GPU crash: restart and run all cells.
- Failure details: inspect `<experiment>/failure.json`; the last valid checkpoint remains intact.
- Clean rerun: move only the relevant experiment directory.
- Never mix smoke-test and full-run artifacts in the same `run_dir`.